In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE

In [2]:
# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("src/Walmart_Store_sales.csv")
print("...Done.")
print()

Loading dataset...
...Done.



In [3]:
# Basic stats
print("Number of rows : {}".format(dataset.shape[0]))
print()

print("Display of dataset: ")
display(dataset.head())
print()

print("Basics statistics: ")
data_desc = dataset.describe(include='all')
display(data_desc)
print()

print("Percentage of missing values: ")
display(100*dataset.isnull().sum()/dataset.shape[0])



Number of rows : 150

Display of dataset: 


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092



Basics statistics: 


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,07-01-2011,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000



Percentage of missing values: 


Store            0.000000
Date            12.000000
Weekly_Sales     9.333333
Holiday_Flag     8.000000
Temperature     12.000000
Fuel_Price       9.333333
CPI              8.000000
Unemployment    10.000000
dtype: float64

In [4]:
print("The following will be 'False' if there's no missing values in the dataset: ", dataset.isnull().any().any())


The following will be 'False' if there's no missing values in the dataset:  True


In [5]:
#gestion des valeurs manquantes#
print("Suppression des NaNs sur Weekly_Sales...")
dataset = dataset.dropna(subset=['Weekly_Sales'])

Suppression des NaNs sur Weekly_Sales...


In [6]:
#feature engineering temporel #
print("Extraction des features de la Date...")

dataset['Date'] = pd.to_datetime(dataset['Date'], format='%d-%m-%Y')
dataset['Year'] = dataset['Date'].dt.year
dataset['Month'] = dataset['Date'].dt.month
dataset['Day'] = dataset['Date'].dt.day
dataset['day_of_week'] = dataset['Date'].dt.dayofweek# 0=Lundi, 6=Dimanche
dataset['hour'] = dataset['Date'].dt.hour

Extraction des features de la Date...


In [7]:
# Suppression de la colonne d'origine#
dataset = dataset.drop(columns=['Date'])

In [8]:
print("Filtrage des Outliers (3 Ecarts-Types)...")
numeric_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

#filtrages des outliers(Règle des 3 écarts-types)
numeric_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

for col in numeric_cols:
    mean_val = dataset[col].mean()
    std_val = dataset[col].std()
    
   # Garde les valeurs normales OU les valeurs manquantes
    mask = (dataset[col].isna()) | ((dataset[col] >= mean_val - 3 * std_val) & (dataset[col] <= mean_val + 3 * std_val))
    dataset = dataset[mask]     

print(f"Dimensions après nettoyage Pandas : {dataset.shape}")


Filtrage des Outliers (3 Ecarts-Types)...
Dimensions après nettoyage Pandas : (131, 12)


In [9]:
print("Imputation des variables numériques par la moyenne...")
dataset = dataset.fillna(dataset.mean(numeric_only=True))
print(f"Dimensions après nettoyage : {dataset.shape}")

Imputation des variables numériques par la moyenne...
Dimensions après nettoyage : (131, 12)


In [10]:
# Séparer la variable cible Y des caractéristiques X

print("Séparation des étiquettes et des caractéristiques...")
target_variable = 'Weekly_Sales'
X = dataset.drop(columns=[target_variable])
Y = dataset[target_variable]
print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Séparation des étiquettes et des caractéristiques...
...Done.

Y : 
0    1572117.54
1    1807545.43
3    1244390.03
4    1644470.66
5    1857533.70
Name: Weekly_Sales, dtype: float64

X :
   Store  Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment  \
0    6.0      0.066667    59.610000    3.045000  214.777523         6.858   
1   13.0      0.000000    42.380000    3.435000  128.616064         7.470   
3   11.0      0.000000    84.570000    3.302908  214.556497         7.346   
4    6.0      0.000000    78.890000    2.759000  212.412888         7.092   
5    4.0      0.000000    60.405897    2.756000  126.160226         7.896   

          Year     Month        Day  day_of_week  hour  
0  2011.000000  2.000000  18.000000          4.0   0.0  
1  2011.000000  3.000000  25.000000          4.0   0.0  
3  2010.831858  6.274336  16.530973          4.0   0.0  
4  2010.000000  5.000000  28.000000          4.0   0.0  
5  2010.000000  5.000000  28.000000          4.0   0.0  


In [11]:
# Diviser le jeu de données en Train set & Test set
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



In [12]:
# separtion des données
target_variable = 'Weekly_Sales'
X = dataset.drop(columns=[target_variable])
Y = dataset[target_variable]

# train_test_split (80% Train, 20% Test)
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=42)

#identifications des types de variables
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day', 'day_of_week']
categorical_features = ['Store', 'Holiday_Flag']

# Create pipeline for numeric features
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # missing values will be replaced by columns' mean
    ('scaler', StandardScaler())
])

# Create pipeline for categorical features
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore'))
])

# Use ColumnTransformer to make a preprocessor object that describes all the treatments to be done
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# Preprocessings on train set
print("Performing preprocessings on train set...")
print(X_train.head())
X_train_processed = preprocessor.fit_transform(X_train)
print('...Done.')
print(X_train[0:5]) # MUST use this syntax because X_train is a numpy array and not a pandas DataFrame anymore
print()

# Preprocessings on test set
print("Performing preprocessings on test set...")
print(X_test.head())
X_test = preprocessor.transform(X_test) # Don't fit again !! The test set is used for validating decisions
# we made based on the training set, therefore we can only apply transformations that were parametered using the training set.
# Otherwise this creates what is called a leak from the test set which will introduce a bias in all your results.
print('...Done.')
print(X_test[0:5,:]) # MUST use this syntax because X_test is a numpy array and not a pandas DataFrame anymore
print()

Performing preprocessings on train set...
    Store  Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment  \
89    2.0      0.000000        76.42       3.786  215.154482         7.931   
51    2.0      0.000000        59.69       2.728  211.660898         8.163   
0     6.0      0.066667        59.61       3.045  214.777523         6.858   
13    1.0      0.000000        64.74       3.734  221.211813         7.348   
46    5.0      0.000000        82.46       2.640  211.927001         6.496   

           Year      Month        Day  day_of_week  hour  
89  2010.831858   6.274336  16.530973          4.0   0.0  
51  2010.000000  11.000000  12.000000          4.0   0.0  
0   2011.000000   2.000000  18.000000          4.0   0.0  
13  2012.000000   3.000000  16.000000          4.0   0.0  
46  2010.000000   7.000000  30.000000          4.0   0.0  
...Done.
    Store  Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment  \
89    2.0      0.000000        76.42       3.786

In [13]:
# Train model
print("Entraînement de la Régression Linéaire...")
regressor = LinearRegression()
regressor.fit(X_train, Y_train)
print("...Done.")

Entraînement de la Régression Linéaire...
...Done.


In [14]:
print("Entraînement de la Régression Linéaire OLS...")
regressor = LinearRegression()
regressor.fit(X_train_processed, Y_train)

Entraînement de la Régression Linéaire OLS...


,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](28,)","[-61242.34,-13754.58, 4116.58,...,453207.3 ,-64624.41, 31174.93]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,1.611e+06
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,28
rank_ rank_: intRank of matrix `X`. Only available when `X` is dense.,int64,np.int64(27)
"singular_ singular_: array of shape (min(X, y),)Singular values of `X`. Only available when `X` is dense.","ndarray[float64](28,)","[14.02,12.42,11.57,..., 0.76, 0.35, 0. ]"


In [15]:
# Predictions on training set
print("Predictions on training set...")
Y_train_pred = regressor.predict(X_train_processed)
print("...Done.")
print(Y_train_pred)
print()


Predictions on training set...
...Done.
[1859598.16295477 2105716.58394762 1503378.4926306  1455497.74599292
  290686.42529274 2028029.61490374  295383.30587302  576042.13490775
 1984476.6658105  1876041.30644586 1972569.27402115 2166617.6614102
 1400871.9795622  1450596.70144753 1608129.90914044 1193976.64724996
 1865922.91906829 1399994.86989848  473872.17333004  823855.94657177
 1035930.10280753 2023222.83698333  498261.03394378 2093814.65819171
  610476.47985181 1363421.84569782  756343.97476403 1718485.00895781
  668796.34448677 1684595.15991461  610836.32422941  501019.98550434
 2002918.82562957  448091.25457289 1253910.6889476  2099427.37561691
 2084551.54461725  187203.88065258 2510176.98128674 1545087.22238069
  316033.29958226  494068.30738506 1462770.58232149  444446.33151694
  956173.94829143 2098071.62815697 1595358.49412794 2131597.42133915
 1675079.13124967 1545775.16877017 1625424.11522783 2032656.12160649
 1573802.60598467  557395.11070253 1081453.0795756  2031712.1374

In [16]:
# Predictions on test set
print("Predictions on test set...")
Y_test_pred = regressor.predict(X_test)
print("...Done.")
print(Y_test_pred)
print()

Predictions on test set...
...Done.
[2079277.73367985 1656134.60304155  205531.1715339  1460373.52090938
 1942331.63897344 1819427.42592013 2124448.54611464 2179739.05962338
  429981.07765041  847150.42798841  997574.25495415 1673917.40520599
 1524753.77296164 2070125.2911318  2369045.0675552  1963514.71354502
 1033349.93514213  803595.51053828 1460855.04510163  907869.77262118
  643971.12168599 1269920.60053815  459972.00770025  590086.45347656
  729042.61667408  373596.2674878   780649.31123757]



In [17]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))


R2 score on training set :  0.9777139215884865
R2 score on test set :  0.8889027368511802


In [18]:
# Perform 3-fold cross-validation to evaluate the generalized R2 score obtained with a Ridge model
print("3-fold cross-validation...")
regressor = Ridge()
scores = cross_val_score(regressor, X_train, Y_train, cv=3)
print('The cross-validated R2-score is : ', scores.mean())
print('The standard deviation is : ', scores.std())


3-fold cross-validation...
The cross-validated R2-score is :  -0.24243818479736667
The standard deviation is :  0.20643506428528294


In [19]:
# Perform grid search
print("Grid search...")
regressor = Ridge()
# Grid of values to be tested
params = {
    'alpha': [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100]
}
best_ridge = GridSearchCV(regressor, param_grid = params, cv = 5) # cv : the number of folds to be used for CV
best_ridge.fit(X_train, Y_train)
print("...Done.")
print("Best hyperparameters : ", best_ridge.best_params_)
print("Best R2 score : ", best_ridge.best_score_)


Grid search...
...Done.
Best hyperparameters :  {'alpha': 100}
Best R2 score :  -0.15936200863664302
